In [27]:
!pip install open3d
!pip install numpy
!pip install plotly
!pip install pandas
!pip install anywidget


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/220.7 kB ? eta -:--:--
   ------- -------------------------------- 41.0/220.7 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 220.7/220.7 kB 4.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/415.6 kB ? eta -:--:--
   ---------------------------------------- 415.6/415.6 kB 8.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [78]:
import os
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

# 1. Define the folder containing your PCD files
data_folder = 'data'
pcd_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.pcd')])

# 2. Define your ground truth translations.
# These are your known, physical measurements of the marker's movement.
# For the first file (datum), the translation is [0, 0, 0].
# Replace the x, y values for the other files with your actual measurements.
ground_truth_translations = {
    pcd_files[0]: np.array([0.0, 0.0, 0.0]),
    pcd_files[1]: np.array([0, 1.2, 0.0]),
    pcd_files[2]: np.array([0.9, 0, 0.0]),
    pcd_files[3]: np.array([0.9, 1.2, 0.0]),
}

print(f"Found {len(pcd_files)} PCD files.")
print(f"Files to process: {pcd_files}")

Found 4 PCD files.
Files to process: ['0,0,0.pcd', '0,1.2,0.pcd', '0.9,0,0.pcd', '0.9,1.2,0.pcd']


In [79]:
# Load the first PCD file
datum_pcd_path = os.path.join(data_folder, pcd_files[0])
datum_pcd = o3d.io.read_point_cloud(datum_pcd_path)

# Visualize the point cloud to find the datum point.
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=np.asarray(datum_pcd.points)[:, 0],
            y=np.asarray(datum_pcd.points)[:, 1],
            z=np.asarray(datum_pcd.points)[:, 2],
            mode='markers',
            marker=dict(size=2, opacity=0.8)
        )
    ]
)
fig.update_layout(
    title=f"Select Datum Point from {pcd_files[0]}",
    scene=dict(
        aspectmode='data'
    ),
    width=1000,
    height=800
)
fig.show()

# Based on your visual inspection and hovering over the point,
# enter the coordinates of the datum point here.
# Replace the placeholder values with the actual x, y, z coordinates you found.
datum_point_pcd = np.array([-0.04256823, 2.371565, -0.5614936])

print("Datum point identified from PCD:", datum_point_pcd)

Datum point identified from PCD: [-0.04256823  2.371565   -0.5614936 ]


### Image fusion for 0,0,0

this is for more accurate point selection

In [25]:
import os
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

# 1. Define the folder containing your PCD files
data_folder = r'C:\Users\Hazimi\OneDrive\CEEM222 MEC698 Internship\ground_calibration\data\0,0,0'
pcd_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.pcd')])

# 2. Create an empty list to hold the point clouds
pcds_to_fuse = []

# 3. Loop through each file, load it, and add it to the list
for filename in pcd_files:
    file_path = os.path.join(data_folder, filename)
    pcd = o3d.io.read_point_cloud(file_path)
    pcds_to_fuse.append(pcd)
    print(f"Loaded {filename} with {len(pcd.points)} points.")

# 4. Combine all the point clouds into a single object
final_fused_pcd = o3d.geometry.PointCloud()
for pcd in pcds_to_fuse:
    final_fused_pcd += pcd  # The '+' operator merges Open3D point clouds

print(f"\nFinal fused point cloud created with {len(final_fused_pcd.points)} points.")

# 5. Visualize the final fused point cloud with a custom camera view
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=np.asarray(final_fused_pcd.points)[:, 0],
            y=np.asarray(final_fused_pcd.points)[:, 1],
            z=np.asarray(final_fused_pcd.points)[:, 2],
            mode='markers',
            marker=dict(size=1, opacity=0.8)
        )
    ]
)

# Customizing the layout and setting the camera view
fig.update_layout(
    title="Fused Point Cloud (View from Y-Axis)",
    scene=dict(
        aspectmode='data',
        xaxis_title='X Axis',
        yaxis_title='Y Axis',
        zaxis_title='Z Axis',
        # Set the default camera position to look along the Y-axis
        camera=dict(
            eye=dict(x=0, y=-2, z=0),  # Position the camera at a negative Y-value
            up=dict(x=0, y=0, z=1)     # Set Z-axis as the 'up' direction
        )
    ),
    width=1000,
    height=800
)

fig.show()

Loaded 0,0,0,2nd.pcd with 36200 points.
Loaded 0,0,0,3rd.pcd with 36200 points.
Loaded 0,0,0,4th.pcd with 36200 points.

Final fused point cloud created with 108600 points.


### Establishing rotation transformation matrix

1. Applying rotation transformation to address the pitch (rotation about x-axis) of the image.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import open3d as o3d
import os
import ipywidgets as widgets
from IPython.display import display
from math import radians
import io
import contextlib

# --- CONFIGURATION ---
# Path to the specific subfolder containing the point cloud data
PCD_FOLDER = r"C:\Users\Hazimi\OneDrive\CEEM222 MEC698 Internship\ground_calibration\data"
PCD_FILE = "0,0,0.pcd"
pcd_path = os.path.join(PCD_FOLDER, PCD_FILE)

# ---- HELPER FUNCTIONS ----

def rotation_matrix(angle_deg, axis='x'):
    """
    Creates a rotation matrix for a given angle and axis.
    """
    angle = np.deg2rad(angle_deg)
    if axis == 'x':
        R = np.array([[1, 0, 0],
                      [0, np.cos(angle), -np.sin(angle)],
                      [0, np.sin(angle), np.cos(angle)]])
    elif axis == 'y':
        R = np.array([[np.cos(angle), 0, np.sin(angle)],
                      [0, 1, 0],
                      [-np.sin(angle), 0, np.cos(angle)]])
    else:  # Roll (Z-axis)
        R = np.array([[np.cos(angle), -np.sin(angle), 0],
                      [np.sin(angle), np.cos(angle), 0],
                      [0, 0, 1]])
    return R

def rotate_about_point(points, pivot_point, angle_deg, axis='x'):
    """
    Rotates a set of points around a specified pivot point.

    Args:
        points (np.ndarray): The array of points to rotate.
        pivot_point (np.ndarray): The single point to rotate around.
        angle_deg (float): The rotation angle in degrees.
        axis (str): The axis to rotate around ('x', 'y', or 'z').

    Returns:
        np.ndarray: The array of rotated points.
        np.ndarray: The 3x3 rotation matrix used.
    """
    # Translate points so pivot is at origin
    translated_points = points - pivot_point
    
    # Get rotation matrix
    R = rotation_matrix(angle_deg, axis)
    
    # Rotate the translated points
    rotated_translated_points = translated_points @ R.T
    
    # Translate points back to original position
    rotated_points = rotated_translated_points + pivot_point
    
    return rotated_points, R

# ---- MAIN SCRIPT EXECUTION ----

try:
    # Load your PCD file
    pcd = o3d.io.read_point_cloud(pcd_path)
    points = np.asarray(pcd.points)
    
    # Find the lowest point (the one with the minimum Z coordinate)
    min_z_idx = np.argmin(points[:, 2])
    lowest_point = points[min_z_idx, :]

    # Create the interactive figure
    fig = go.FigureWidget()

    # Add the original point cloud trace (for reference)
    fig.add_trace(go.Scatter3d(
        x=points[:,0], y=points[:,1], z=points[:,2],
        mode='markers',
        marker=dict(size=1.5, color='gray', opacity=0.5),
        name='Original Point Cloud'
    ))

    # Add the leveled point cloud trace (will be updated by the slider)
    fig.add_trace(go.Scatter3d(
        x=points[:,0], y=points[:,1], z=points[:,2],
        mode='markers',
        marker=dict(size=1.5, color='blue', opacity=0.8),
        name=f'Leveled Point Cloud'
    ))

    # Add the lowest point as a red marker
    fig.add_trace(go.Scatter3d(
        x=[lowest_point[0]], y=[lowest_point[1]], z=[lowest_point[2]],
        mode='markers',
        marker=dict(size=5, color='red'),
        name='Lowest Point (Pivot)'
    ))
    
    # Layout for the plot with a default view along the X-axis
    fig.update_layout(
        title="Point Cloud Leveled About Lowest Point (X-axis)",
        scene=dict(
            aspectmode='data',
            xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
            camera=dict(
                up=dict(x=0, y=0, z=1),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=-2.5, y=0, z=0)  # View along Y-Z plane
            )
        ),
        showlegend=True,
        width=1000,
        height=800
    )

    # Create the slider
    angle_slider = widgets.FloatSlider(
        value=0.0,
        min=-90,
        max=90,
        step=0.5, # The step is already 0.5 as requested
        description='Angle (deg):',
        continuous_update=True
    )
    
    # Create an output widget to display the matrix
    output = widgets.Output()

    # Function to update the plot when the slider changes
    def update_plot(change):
        with output:
            output.clear_output()
            new_angle = change['new']
            leveled_points, rotation_matrix = rotate_about_point(points, lowest_point, new_angle, axis='x')
            
            with fig.batch_update():
                fig.data[1].x = leveled_points[:,0]
                fig.data[1].y = leveled_points[:,1]
                fig.data[1].z = leveled_points[:,2]
                fig.data[1].name = f'Leveled Point Cloud ({new_angle}°)'
                
            print(f"\n--- Rotation Matrix for Angle: {new_angle} ---")
            print(np.round(rotation_matrix, 4))
            print("--------------------------------------------------\n")

    # Connect the slider to the update function
    angle_slider.observe(update_plot, names='value')

    # Display the widgets and the plot
    display(widgets.VBox([angle_slider, fig, output]))

except FileNotFoundError:
    print(f"Error: The file '{pcd_path}' was not found.")
    print("Please check the PCD_FOLDER and PCD_FILE variables to ensure they point to a valid file.")

To apply the levelling/rotation matrix, run the apply_leveling_matrix.py with appropriate value according to the rotation matrix.

# PCD VIEWER
View PCD image data here

### Image for rotation, before translation applied

In [82]:
import numpy as np
import open3d as o3d
import plotly.graph_objs as go

def view_pcd(path, point_size=2, color_mode='z', view='xyz'):
    """
    Quick .pcd viewer in Jupyter using Plotly.
    color_mode: 'z' | 'rgb' | 'gray'
    view: 'xyz' (default 3D), 'yz', or 'xz'
    """
    # load PCD
    pcd = o3d.io.read_point_cloud(path)
    pts = np.asarray(pcd.points)
    if pts.size == 0:
        raise ValueError("No points in PCD.")
    xs, ys, zs = pts[:,0], pts[:,1], pts[:,2]

    # colors
    if color_mode == 'rgb' and pcd.has_colors():
        rgb = (np.asarray(pcd.colors) * 255).astype(np.uint8)
        marker_color = ['rgb(%d,%d,%d)' % tuple(c) for c in rgb]
        marker = dict(size=point_size, color=marker_color)
    elif color_mode == 'z':
        marker = dict(size=point_size, color=zs, colorscale='Viridis', showscale=True)
    else:
        marker = dict(size=point_size, color='gray')

    trace = go.Scatter3d(x=xs, y=ys, z=zs, mode='markers', marker=marker)

    # default camera orientations
    camera = dict()
    if view == 'yz':  # look along x-axis
        camera = dict(eye=dict(x=2.5, y=0., z=0.))
    elif view == 'xz':  # look along y-axis
        camera = dict(eye=dict(x=0., y=2.5, z=0.))

    fig = go.Figure(data=[trace])
    fig.update_layout(scene=dict(aspectmode='data'),
                      margin=dict(l=0, r=0, b=0, t=30),
                      scene_camera=camera)
    fig.show()


In [85]:
view_pcd(r"data\Sample\leveled_data\0,0,0.pcd",
         point_size=1, color_mode='rgb', view='yz')

### Image for Rotation + Translation

In [86]:
import numpy as np
import open3d as o3d
import plotly.graph_objs as go

def view_pcd(path, point_size=2, color_mode='z', view='xyz'):
    """
    Quick .pcd viewer in Jupyter using Plotly.
    color_mode: 'z' | 'rgb' | 'gray'
    view: 'xyz' (default 3D), 'yz', or 'xz'
    """
    # load PCD
    pcd = o3d.io.read_point_cloud(path)
    pts = np.asarray(pcd.points)
    if pts.size == 0:
        raise ValueError("No points in PCD.")
    xs, ys, zs = pts[:,0], pts[:,1], pts[:,2]

    # colors
    if color_mode == 'rgb' and pcd.has_colors():
        rgb = (np.asarray(pcd.colors) * 255).astype(np.uint8)
        marker_color = ['rgb(%d,%d,%d)' % tuple(c) for c in rgb]
        marker = dict(size=point_size, color=marker_color)
    elif color_mode == 'z':
        marker = dict(size=point_size, color=zs, colorscale='Viridis', showscale=True)
    else:
        marker = dict(size=point_size, color='gray')

    trace = go.Scatter3d(x=xs, y=ys, z=zs, mode='markers', marker=marker)

    # default camera orientations
    camera = dict()
    if view == 'yz':  # look along x-axis
        camera = dict(eye=dict(x=2.5, y=0., z=0.))
    elif view == 'xz':  # look along y-axis
        camera = dict(eye=dict(x=0., y=2.5, z=0.))

    fig = go.Figure(data=[trace])
    fig.update_layout(scene=dict(aspectmode='data'),
                      margin=dict(l=0, r=0, b=0, t=30),
                      scene_camera=camera)
    fig.show()


In [87]:
view_pcd(r"data\Sample\leveled_data\final_grounded.pcd",
         point_size=1, color_mode='rgb', view='yz')

#### 1. Load point cloud data for preview

In [54]:
import numpy as np
import open3d as o3d

# Load your point cloud
pcd = o3d.io.read_point_cloud(r"data\Sample\leveled_data\0,0,0.pcd")
points = np.asarray(pcd.points)

# Show first few points
points[:5]


array([[2.16237426, 3.33002019, 0.32714298],
       [2.13075995, 3.33419633, 0.3361035 ],
       [2.09544611, 3.33111191, 0.34505832],
       [2.05775881, 3.32468557, 0.35444224],
       [2.02284718, 3.32080078, 0.36331367]])

#### 2. Define rotation matrix and new datum point

In [65]:
# Rotation matrix (your leveling calibration)
R = np.array([
    [1.0000, 0.0000, 0.0000],
    [0.0000, 0.95372, 0.300706],
    [0.0000, -0.300706, 0.95372]
])

# New datum coordinates you want to assign
P_datum = np.array([-0.05546083, 2.369946, -0.5661217]) # this value can be different based on the corner determination that is done manually

# Compute translation vector to bring datum to origin
t = -P_datum


#### 3. Built the homogenous transformation matrix

In [66]:
# Build 4x4 homogeneous matrix
T = np.eye(4)
T[:3, :3] = R
T[:3, 3] = t

T


array([[ 1.        ,  0.        ,  0.        ,  0.05546083],
       [ 0.        ,  0.95372   ,  0.300706  , -2.369946  ],
       [ 0.        , -0.300706  ,  0.95372   ,  0.5661217 ],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

#### 4. Convert points to homogenous coordinates

In [67]:
# Convert points to Nx4
points_h = np.hstack([points, np.ones((points.shape[0],1))])

#### 5. Apply rotation + translation

In [72]:
points_transformed = (T @ points_h.T).T[:, :3]


#### 6. Saved transformed cloud

In [74]:
pcd_transformed = o3d.geometry.PointCloud()
pcd_transformed.points = o3d.utility.Vector3dVector(points_transformed)
o3d.io.write_point_cloud("data/Sample/leveled_data/final_grounded.pcd", pcd_transformed)

True

### 7. Numerical verification

In [75]:
# Check that the datum is now at origin
idx = np.argmin(points[:, 2])  # or the index of the chosen point
print("Transformed datum coordinates:", points_transformed[idx])

Transformed datum coordinates: [ 0.09030132 -0.29403576 -0.74049258]


#### 8. Visual Verification

In [84]:
o3d.visualization.draw_geometries([pcd_transformed])